In [1]:
import itertools
import numpy as np
import pandas as pd
from dtw import dtw

Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



In [2]:
def make_sections(dfs, section_names, params, skips=None):
    """
    Extract multivariate arrays from Excel sheets.

    Returns
    -------
    sections : dict
        {section_name: array of shape (n_points, n_params)}
    """
    if skips is None:
        skips = {}

    sections = {}

    for name in section_names:
        skip = skips.get(name, 0)

        arr = (
            dfs[name][params]
            .iloc[skip:]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=float)
        )

        sections[name] = arr

    return sections

In [3]:
def make_weight_vector(params, weights=None):
    """
    Convert either a dict or a list/array of weights to a numpy vector.
    """
    if weights is None:
        return np.ones(len(params), dtype=float)

    if isinstance(weights, dict):
        return np.array([weights.get(p, 1.0) for p in params], dtype=float)

    weights = np.asarray(weights, dtype=float)

    if len(weights) != len(params):
        raise ValueError("weights must have the same length as params.")

    return weights


In [4]:
def make_circular_mask(params, circular_params=("Dgeo",)):
    """
    Mark circular variables, e.g. declination in degrees.
    """
    circular_params = set(circular_params)
    return np.array([p in circular_params for p in params], dtype=bool)

In [5]:
def compute_global_scales(
    sections,
    params=None,
    method="mad",
    scale_overrides=None,
    min_scale=1e-12,
):
    """
    Estimate one scale per parameter across all sections.

    method="mad" is robust.
    method="std" uses standard deviation.

    scale_overrides can be used for known measurement precision, e.g.
    {"Dgeo": 15.0, "Igeo": 10.0}
    """
    all_values = np.vstack(list(sections.values()))

    if method == "mad":
        center = np.nanmedian(all_values, axis=0)
        mad = np.nanmedian(np.abs(all_values - center), axis=0)
        scales = 1.4826 * mad
    elif method == "std":
        scales = np.nanstd(all_values, axis=0)
    else:
        raise ValueError("method must be 'mad' or 'std'.")

    scales = np.asarray(scales, dtype=float)
    scales[~np.isfinite(scales)] = 1.0
    scales[scales < min_scale] = 1.0

    if scale_overrides is not None:
        if params is None:
            raise ValueError("params must be provided when using scale_overrides.")

        for param_name, scale_value in scale_overrides.items():
            if param_name not in params:
                raise ValueError(f"{param_name!r} is not in params.")
            scales[params.index(param_name)] = float(scale_value)

    return scales

In [6]:
def circular_difference_degrees(a, b):
    """
    Smallest signed angular difference a - b for degrees.
    Result is in [-180, 180].
    """
    return (a - b + 180.0) % 360.0 - 180.0

In [7]:
def weighted_nan_dist(
    a,
    b,
    weights=None,
    scales=None,
    circular=None,
    metric="l1",
    missing_penalty=0.0,
    empty_penalty=1e6,
):
    """
    Weighted multivariate local distance for DTW.

    Important behavior:
    - unit differences are handled by scales;
    - scientific trust is handled by weights;
    - NaNs are ignored when at least one common parameter exists;
    - if no common parameter exists, empty_penalty is returned.

    metric:
    - "l1": weighted mean absolute normalized difference
    - "l2": weighted root-mean-square normalized difference
    """
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()
    n = len(a)

    if weights is None:
        weights = np.ones(n, dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)

    if scales is None:
        scales = np.ones(n, dtype=float)
    else:
        scales = np.asarray(scales, dtype=float)

    if circular is None:
        circular = np.zeros(n, dtype=bool)
    else:
        circular = np.asarray(circular, dtype=bool)

    if not (len(b) == len(weights) == len(scales) == len(circular) == n):
        raise ValueError("a, b, weights, scales, and circular must have compatible lengths.")

    valid = (
        np.isfinite(a)
        & np.isfinite(b)
        & np.isfinite(weights)
        & np.isfinite(scales)
        & (weights > 0)
        & (scales > 0)
    )

    if not np.any(valid):
        return float(empty_penalty)

    diff = a - b

    if np.any(circular & valid):
        diff[circular & valid] = circular_difference_degrees(
            a[circular & valid],
            b[circular & valid],
        )

    z = np.abs(diff[valid]) / scales[valid]
    w = weights[valid]

    if metric == "l1":
        d = np.sum(w * z) / np.sum(w)
    elif metric == "l2":
        d = np.sqrt(np.sum(w * z * z) / np.sum(w))
    else:
        raise ValueError("metric must be 'l1' or 'l2'.")

    if missing_penalty > 0:
        missing = ~valid
        missing_weight_fraction = np.sum(weights[missing]) / np.sum(weights)
        d += missing_penalty * missing_weight_fraction

    return float(d)

In [8]:
def prepare_anchors(anchors, n_x, n_y, add_endpoints=True):
    """
    Validate anchors for x and y.

    Anchors must be pairs of post-skip indices:
        [(x_index, y_index), ...]

    They must be monotonic in both x and y.
    """
    if anchors is None:
        anchors = []

    anchors = [(int(i), int(j)) for i, j in anchors]

    if add_endpoints:
        if (0, 0) not in anchors:
            anchors.append((0, 0))
        if (n_x - 1, n_y - 1) not in anchors:
            anchors.append((n_x - 1, n_y - 1))

    anchors = sorted(set(anchors), key=lambda t: (t[0], t[1]))

    for i, j in anchors:
        if i < 0 or i >= n_x or j < 0 or j >= n_y:
            raise ValueError(
                f"Anchor {(i, j)} is outside array sizes {(n_x, n_y)}."
            )

    for (i0, j0), (i1, j1) in zip(anchors[:-1], anchors[1:]):
        if i1 <= i0 or j1 <= j0:
            raise ValueError(
                "Anchors must be strictly increasing in both sections. "
                f"Problem: {(i0, j0)} -> {(i1, j1)}"
            )

    return anchors

In [9]:
def get_pair_anchors(name_x, name_y, n_x, n_y, anchors_by_pair=None):
    """
    Fetch anchors for a pair of sections.

    anchors_by_pair example:
    {
        ("C", "B"): [(0, 0), (4, 3), (15, 13)]
    }

    If a reversed key exists, anchors are automatically swapped.
    If no anchors are provided, only endpoints are used.
    """
    if anchors_by_pair is None:
        anchors = None
    elif (name_x, name_y) in anchors_by_pair:
        anchors = anchors_by_pair[(name_x, name_y)]
    elif (name_y, name_x) in anchors_by_pair:
        anchors = [(j, i) for i, j in anchors_by_pair[(name_y, name_x)]]
    else:
        anchors = None

    return prepare_anchors(anchors, n_x=n_x, n_y=n_y, add_endpoints=True)

In [14]:
def anchored_dtw_multi(
    x,
    y,
    anchors=None,
    weights=None,
    scales=None,
    circular=None,
    metric="l1",
    missing_penalty=0.0,
    empty_penalty=1e6,
    normalize_distance=True,
    **dtw_kwargs,
):
    """
    Anchored multivariate DTW between two sections.

    Returns
    -------
    result : dict
        index_x : indices in x along final warping path
        index_y : indices in y along final warping path
        distance : recomputed final path distance, without duplicated anchor costs
        normalized_distance : distance / path_length
        path_length : number of matched pairs in final path
        anchors : validated anchor list
        dtw_objective_distance : raw sum of DTW segment objective distances
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if x.ndim == 1:
        x = x.reshape(-1, 1)

    if y.ndim == 1:
        y = y.reshape(-1, 1)

    if x.shape[1] != y.shape[1]:
        raise ValueError("x and y must have the same number of parameters.")

    anchors = prepare_anchors(
        anchors,
        n_x=len(x),
        n_y=len(y),
        add_endpoints=True,
    )

    def dist_method(a, b):
        return weighted_nan_dist(
            a,
            b,
            weights=weights,
            scales=scales,
            circular=circular,
            metric=metric,
            missing_penalty=missing_penalty,
            empty_penalty=empty_penalty,
        )

    full_index_x = []
    full_index_y = []
    dtw_objective_distance = 0.0

    for (i_start, j_start), (i_end, j_end) in zip(anchors[:-1], anchors[1:]):
        x_seg = x[i_start : i_end + 1]
        y_seg = y[j_start : j_end + 1]

        alignment = dtw(
            x_seg,
            y_seg,
            dist_method=dist_method,
            keep_internals=True,
            **dtw_kwargs,
        )

        idx_x = np.asarray(alignment.index1, dtype=int) + i_start
        idx_y = np.asarray(alignment.index2, dtype=int) + j_start

        # Avoid duplicate anchor pair at segment boundary.
        if len(full_index_x) > 0:
            idx_x = idx_x[1:]
            idx_y = idx_y[1:]

        full_index_x.extend(idx_x)
        full_index_y.extend(idx_y)

        # Keep this only as diagnostic information.
        dtw_objective_distance += float(alignment.distance)

    full_index_x = np.asarray(full_index_x, dtype=int)
    full_index_y = np.asarray(full_index_y, dtype=int)

    path_length = len(full_index_x)

    # Recompute the actual path distance after duplicate anchor removal.
    path_costs = [
        dist_method(x[i], y[j])
        for i, j in zip(full_index_x, full_index_y)
    ]

    distance = float(np.sum(path_costs))

    if normalize_distance:
        normalized_distance = distance / max(path_length, 1)
    else:
        normalized_distance = distance

    return {
        "index_x": full_index_x,
        "index_y": full_index_y,
        "distance": distance,
        "normalized_distance": normalized_distance,
        "path_length": path_length,
        "anchors": anchors,
        "dtw_objective_distance": dtw_objective_distance,
    }

In [11]:
def pairwise_dtw_matrix(
    sections,
    params,
    weights=None,
    scales=None,
    anchors_by_pair=None,
    circular_params=("Dgeo",),
    metric="l1",
    missing_penalty=0.0,
    empty_penalty=1e6,
    **dtw_kwargs,
):
    """
    Compute all-vs-all multivariate DTW distances.

    Returns
    -------
    distance_matrix : pandas.DataFrame
        Symmetric matrix of normalized DTW distances.

    alignments : dict
        {(name_x, name_y): alignment_result}
    """
    names = list(sections.keys())

    weights = make_weight_vector(params, weights)

    if scales is None:
        scales = compute_global_scales(sections, params=params, method="mad")

    circular = make_circular_mask(params, circular_params=circular_params)

    distance_matrix = pd.DataFrame(
        np.zeros((len(names), len(names))),
        index=names,
        columns=names,
        dtype=float,
    )

    alignments = {}

    for name_x, name_y in itertools.combinations(names, 2):
        x = sections[name_x]
        y = sections[name_y]

        anchors = get_pair_anchors(
            name_x,
            name_y,
            n_x=len(x),
            n_y=len(y),
            anchors_by_pair=anchors_by_pair,
        )

        result = anchored_dtw_multi(
            x,
            y,
            anchors=anchors,
            weights=weights,
            scales=scales,
            circular=circular,
            metric=metric,
            missing_penalty=missing_penalty,
            empty_penalty=empty_penalty,
            **dtw_kwargs,
        )

        d = result["normalized_distance"]

        distance_matrix.loc[name_x, name_y] = d
        distance_matrix.loc[name_y, name_x] = d

        alignments[(name_x, name_y)] = result

    return distance_matrix, alignments

In [12]:
def choose_dtw_medoid(distance_matrix):
    """
    Choose the section with the smallest mean distance to all other sections.
    """
    distance_matrix = distance_matrix.copy()
    np.fill_diagonal(distance_matrix.values, np.nan)

    mean_distance = distance_matrix.mean(axis=1, skipna=True)
    medoid_name = mean_distance.idxmin()

    return medoid_name, mean_distance


In [13]:
def align_all_to_reference(
    sections,
    reference_name,
    params,
    weights=None,
    scales=None,
    anchors_by_pair=None,
    circular_params=("Dgeo",),
    metric="l1",
    missing_penalty=0.0,
    empty_penalty=1e6,
    **dtw_kwargs,
):
    """
    Align every section to one reference section.

    Output alignment for each non-reference section has:
    - index_x = indices in reference
    - index_y = indices in the aligned section
    """
    if reference_name not in sections:
        raise ValueError(f"{reference_name!r} is not in sections.")

    weights = make_weight_vector(params, weights)

    if scales is None:
        scales = compute_global_scales(sections, params=params, method="mad")

    circular = make_circular_mask(params, circular_params=circular_params)

    ref = sections[reference_name]
    alignments_to_ref = {}

    for name, arr in sections.items():
        if name == reference_name:
            continue

        anchors = get_pair_anchors(
            reference_name,
            name,
            n_x=len(ref),
            n_y=len(arr),
            anchors_by_pair=anchors_by_pair,
        )

        result = anchored_dtw_multi(
            ref,
            arr,
            anchors=anchors,
            weights=weights,
            scales=scales,
            circular=circular,
            metric=metric,
            missing_penalty=missing_penalty,
            empty_penalty=empty_penalty,
            **dtw_kwargs,
        )

        alignments_to_ref[name] = result

    return alignments_to_ref

In [19]:
xl_file = pd.ExcelFile('I_full data.xlsx')

dfs = {sheet_name: xl_file.parse(sheet_name) 
          for sheet_name in xl_file.sheet_names}

In [20]:
depth_min = 0
depth_max = 120

In [21]:
dfs['B'].columns

Index(['thickness, mm', 'Piece', 'sample №', 'thick, mm', 'weight, g',
       'Vol, cm3', 'depth, mm', 'Dgeo', 'Igeo', 'MADgeo', 'NRMmax (Am2/kg)',
       'ARMmax (Am2/kg)'],
      dtype='str')

In [40]:
section_names = [
    "A",
    "B",
    "C",
    "D",
    "E",
    "F",
]

params = [
    "Dgeo",
    "Igeo",
    #"NRMmax (Am2/kg)",
    #"ARMmax (Am2/kg)",
]

In [41]:
sections = make_sections(
    dfs=dfs,
    section_names=section_names,
    params=params,
    skips=skips,
)

In [42]:
for name, arr in sections.items():
    print(name, arr.shape, "NaNs:", np.isnan(arr).sum())

A (37, 2) NaNs: 14
B (36, 2) NaNs: 6
C (34, 2) NaNs: 6
D (33, 2) NaNs: 6
E (33, 2) NaNs: 10
F (29, 2) NaNs: 2


In [43]:
weights = {
    "Dgeo": 0.25,
    "Igeo": 1.0,
    "NRMmax (Am2/kg)": 0.25,
    "ARMmax (Am2/kg)": 1.0,
}

In [44]:
scales = compute_global_scales(
    sections,
    params=params,
    method="mad",
    scale_overrides={
        "Dgeo": 20.0,
        "Igeo": 10.0,
    },
)

pd.Series(scales, index=params)

Dgeo    20.0
Igeo    10.0
dtype: float64

In [45]:
distance_matrix, pair_alignments = pairwise_dtw_matrix(
    sections=sections,
    params=params,
    weights=weights,
    scales=scales,
    anchors_by_pair=None,
    circular_params=("Dgeo",),
    metric="l1",
    missing_penalty=0.0,
    step_pattern="symmetric2",
)

In [46]:
distance_matrix

,A,B,C,D,E,F
A,0.000000,166667.740296,166667.671889,169812.273321,200001.410564,140001.726760
B,166667.740296,0.000000,96155.048077,94340.599189,134616.868365,58825.281843
C,166667.671889,96155.048077,0.000000,120000.901220,153847.747192,60001.733200
D,169812.273321,94340.599189,120000.901220,0.000000,140001.479060,80001.889020
E,200001.410564,134616.868365,153847.747192,140001.479060,0.000000,127660.998128
F,140001.726760,58825.281843,60001.733200,80001.889020,127660.998128,0.000000
